# Notebook 02 — Linear Models, Loss, and Gradient Descent

In the previous notebook we established that a model is just a function $f(x_1, \dots, x_n)$ whose shape we learn from data. Remember that the inputs $x_1, \dots, x_n$ are called features. Previously, we considered the example $f(x_1, x_2) = 3 x_1 + 2 x_2$.

Models of such shape are called **linear** models because they draw a straight line (just like how $y = 2x + 1$ is a line). In this notebook we'll make linear model precise, turn loss into an actual formula, and finally meet **gradient descent**: the procedure a computer really uses to train a model.

**Note 1:** Don't worry about understanding the code.  
**Note 2:** This notebook is longer and more technical than the previous one. Don't get frustrated!

## Linear models are lines

The simplest linear function of one input looks like $y = 2x + 1$. We get a perfectly straight line when we plot it:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.linspace(-2, 3, 100)

m = 2
b = 1
y = m * x + b

plt.figure(figsize=(5, 4))
plt.plot(x, y, color="tab:red", lw=2, label=f"y = {m}x + {b}")  # only this is labeled

x_intercept = -b / m
y_intercept = b

plt.scatter([x_intercept], [0], color="black", zorder=5)
plt.scatter([0], [y_intercept], color="black", zorder=5)
plt.annotate(f"({x_intercept:.2f}, 0)", xy=(x_intercept, 0), xytext=(8, -14),
             textcoords="offset points", ha="left", va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.8))
plt.annotate(f"(0, {y_intercept:.2f})", xy=(0, y_intercept), xytext=(8, 6),
             textcoords="offset points", ha="left", va="bottom", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.8))

plt.axhline(0, color="black", lw=0.5)
plt.axvline(0, color="black", lw=0.5)
plt.xlabel("x"); plt.ylabel("y")
plt.legend()
plt.title("A linear model")
plt.show()

## The general form: weights and bias

Let's generalize. We call a model **linear** if it has this form:

$$ f(x_1, \dots, x_n) = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n $$

The numbers $w_0, w_1, \dots, w_n$ are called the **weights**. Reading the output as a weighted sum, each weight $w_i$ says how much its feature $x_i$ matters. For instance, if house prices in some fictitious city depended only on $x_1$ (square footage), we could set $w_2 = \dots = w_n = 0$. Every other feature gets multiplied by zero, so they are simply ignored. Then the model becomes $f(x_1, \dots, x_n) = w_0 + w_1 x_1$, which we just saw is a line. Notice that $w_0$ does not multiply any feature. It is a constant, added no matter what the input is. That term is also called the **bias**. It shifts the whole line up (or down) by a fixed offset with respect to the origin.

This also hints that there are **non-linear** models as well! In fact, most real-world tasks (like image segmentation) are ultimately non-linear. But linear models are the perfect place to build intuition, so we'll stay with them for now.

## Back to the house-price example

Let's bring back the exact same dataset from the previous notebook: 10 houses, each with a size (our single feature $x$) and a price (the value $y$ we want to predict).

In [ ]:
rng = np.random.default_rng(1)
sqft  = np.sort(rng.integers(600, 3000, size=10))
price = 50_000 + 150 * sqft + rng.normal(0, 30_000, size=sqft.shape)

plt.figure(figsize=(6, 4))
plt.scatter(sqft, price, color="tab:blue", alpha=0.8)
plt.xlabel("size (square feet)")
plt.ylabel("price (dollars)")
plt.title("House-price dataset")
plt.show()

In [ ]:
import pandas as pd
dataset = pd.DataFrame({"size (sq ft)": sqft, "price ($)": price.round().astype(int)})
dataset = dataset.sort_values("size (sq ft)").reset_index(drop=True)
dataset.index = dataset.index + 1
dataset.index.name = "house"
dataset

## Fitting a line to the houses

We have a single feature (size, which we'll just call $x$), so our linear model is

$$ f(x) = w_0 + w_1 x $$

A bias $w_0$ and one weight $w_1$. Let's fit this model to the data, that is, find the $w_0$ and $w_1$ that make the line pass as close to the points as possible.

In [ ]:
w1, w0 = np.polyfit(sqft, price, 1)
print("fitted weight  w1 =", round(float(w1), 2), "dollars per square foot")
print("fitted bias    w0 =", round(float(w0)))

def f(x):
    return w0 + w1 * x

xs = np.linspace(500, sqft.max(), 100)
plt.figure(figsize=(6, 4))
plt.scatter(sqft, price, color="tab:blue", alpha=0.7, label="Training data")
plt.plot(xs, f(xs), color="tab:red", lw=2, label=f"fitted line  f(x) = {w0:.0f} + {w1:.2f} x")
plt.xlabel("Size (square feet)"); plt.ylabel("Price (dollars)")
plt.legend()
plt.title("A fitted linear model")
plt.show()

## What does "good fit" mean to a computer?

Good! To the human eye, the fitted model (the red line) aligns well with the data points. But let's remember, from the computer's point of view, a good fit is when we have low loss.

Previously, we mentioned that loss measures how far off a model’s predictions are on average, and that it is typically some function of the errors across all data points. One of the most common choices is the **squared error**. Writing $y$ for the true value (the price in our dataset) and $\hat{y}$ ("y-hat") for the model's prediction:

$$ \mathcal{L}(y, \hat{y}) = (y - \hat{y})^2 $$

In words: take the truth $y$, subtract the prediction $\hat{y}$ to get the error $y - \hat{y}$, then square it. Squaring makes the loss positive (an error of $-3$ is just as bad as $+3$) and punishes big mistakes far more than small ones.

The squared loss scores a single point. To score the model on the whole dataset of $N$ examples (in our house-prices dataset, $N = 10$), we average the squared errors. This is the famous **Mean Squared Error (MSE)**:

$$ \mathcal{L}_{\text{MSE}} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 $$

where $y_i$ is the true price of the $i$-th house and $\hat{y}_i = f(x_i)$ is our model's prediction for it. It is exactly what it says: the mean (average) of the squared errors across every example in the dataset.

Let's compute the MSE of the line we just fitted:

In [ ]:
y_true = price
y_pred = f(sqft)
diff   = y_true - y_pred

report = pd.DataFrame({
    "size (sq ft)":    sqft,
    "actual price":    y_true.round().astype(int),
    "predicted price": y_pred.round().astype(int),
    "difference":      diff.round().astype(int),
    "squared error":   (diff ** 2).round().astype(int),
})
report = report.sort_values("size (sq ft)").reset_index(drop=True)
report.index = report.index + 1
report.index.name = "house"

mse = np.mean(diff ** 2)
print("Mean Squared Error (MSE) =", round(float(mse)))
print("As a rough feel, the typical miss is about", round(float(np.sqrt(mse))), 
      "dollars (square root of the MSE, aka root mean squared error (RMSE)).")
report

## How does a computer find the best weights?

So far, we've been relying on our magical tool to find the best weights for us ("best" that minimizes the loss). But how would a machine discover the right weights on its own? In order to get a sense, first let's train a model by hand. 

Below are **two sliders**: one for the weight $w_1$ (the slope) and one for the bias $w_0$ (the up/down shift). As you drag them:
- Each blue point grows an orange stick showing how far the line misses it. That gap is the error for that point.
- On the right panel, you'll see how the MSE changes with every move.

Try to make the line fit the points, i.e. make the MSE as small as you can. Watch the curve on the right head downward. You are doing by hand exactly what training does automatically: searching for the weights with the lowest loss.

In [ ]:
import io

line_x = np.linspace(0, 3000, 100)

def mse_at(w1_try, w0_try):
    return float(np.mean((price - (w0_try + w1_try * sqft)) ** 2))

def redraw_axes(axL, axR, w1_try, w0_try, history):
    mse  = history[-1]
    rmse = np.sqrt(mse)

    axL.clear()
    pred = w0_try + w1_try * sqft
    axL.vlines(sqft, np.minimum(price, pred), np.maximum(price, pred),
               color="tab:purple", lw=2, label="Error per house")
    axL.scatter(sqft, price, color="tab:blue", zorder=3, label="Data")
    axL.plot(line_x, w0_try + w1_try * line_x, color="tab:red", lw=2, label="Your model")
    axL.axvline(0, color="gray", lw=0.8, ls=":")
    axL.set_xlim(0, 3000); axL.set_ylim(0, 950_000)
    axL.set_xlabel("Size (square feet)"); axL.set_ylabel("Price (dollars)")
    axL.legend(loc="upper left")
    readout = f"MSE  = {mse:,.0f}" + chr(10) + f"RMSE = {rmse:,.0f}"
    axL.text(0.97, 0.04, readout, transform=axL.transAxes, ha="right", va="bottom",
             family="monospace", fontsize=11,
             bbox=dict(boxstyle="round", facecolor="whitesmoke", edgecolor="gray"))

    axR.clear()
    steps = range(1, len(history) + 1)
    axR.plot(steps, history, color="tab:purple", lw=2, marker="o", ms=3)
    axR.scatter([len(history)], [mse], color="black", zorder=5, label="Current loss")
    axR.set_xlabel("Loss history"); axR.set_ylabel("loss (MSE)")
    axR.set_title("Mean squared loss", fontsize=10)
    axR.legend(loc="upper right", fontsize=8)

def render_png(fig):
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    return buf.getvalue()

import ipywidgets as widgets
from IPython.display import display

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.3),
                                gridspec_kw={"width_ratios": [2, 1]})
plt.close(fig)

history = []
canvas  = widgets.Image(format="png")

slider = dict(continuous_update=True, style={"description_width": "initial"},
                layout=widgets.Layout(width="380px"))
w1_slider = widgets.FloatSlider(value=200, min=0, max=300, step=5,
                                description="weight w1", readout_format=".0f", **slider)
w0_slider = widgets.FloatSlider(value=50000, min=0, max=150_000, step=5_000,
                                description="bias w0", readout_format=",.0f", **slider)

def redraw(_=None):
    history.append(mse_at(w1_slider.value, w0_slider.value))
    del history[:-50]
    redraw_axes(axL, axR, w1_slider.value, w0_slider.value, history)
    canvas.value = render_png(fig)

w1_slider.observe(redraw, names="value")
w0_slider.observe(redraw, names="value")
redraw()
display(widgets.VBox([widgets.HBox([w1_slider, w0_slider]), canvas]))

Did you notice yourself moving the slider in one direction, watching the MSE decrease, then (once it started climbing back up) reversing course to bring it down again? Hold on to that — that's the core of the **gradient descent** algorithm we're about to discuss.

## A quick word: partial derivatives
Before we get to the gradient descent algorithm, we need to understand what a **partial derivative** is. 

Remember that a regular derivative essentially tells you how fast a function's output changes as you nudge its input. Let's consider this function:
$$ f(x) = x^2 + 1; \qquad f'(x) = 2x $$

In [ ]:
x_ref = np.linspace(-3, 3, 200)
f_ref = x_ref ** 2 + 1
f_prime_ref = 2 * x_ref

plt.figure(figsize=(6, 4))
plt.plot(x_ref, f_ref, color="tab:red", lw=2, label="f(x) = x^2 + 1")
plt.plot(x_ref, f_prime_ref, color="tab:green", lw=2, label="f'(x) = 2x")
plt.axhline(0, color="black", lw=0.5)
plt.axvline(0, color="black", lw=0.5)
plt.xlabel("x"); plt.ylabel("value")
plt.legend()
plt.title("f(x) and f'(x)")
plt.show()

Let's check the behavior of $f$ around $x=3$. Define a small $\Delta x = 0.001$ and compute $f(x)$ and $f(x + \Delta x)$: 
$$\begin{aligned}
f(3) &= 3^2 + 1 = 10 \\
f(3 + \Delta x) &= (3 + 0.001)^2 + 1 = 10.006
\end{aligned}$$

Remember that what derivative tells you is this: *"If I change $x$ by $\Delta x$, then $f(x)$ will change (roughly) by the **derivative at that point times $\Delta x$**."* Let's check that:

$$\begin{aligned}
\Delta f = f(3 + \Delta x) - f(3) = 0.006\\
f'(3) \cdot \Delta x = 6 \cdot 0.001 = 0.006
\end{aligned}$$

So the numbers indeed support this claim. That's one key point that the gradient descent algorithm relies on. It's also sometimes useful to think of the derivative as a slope of the function at a given point. The steeper the slope, the faster the function's output changes as you move along the input axis.

Moreover, notice that the derivative encodes a *direction* as well. If we instead looked at $x=-3$, the derivative would be $f'(-3) = -6$. The negative sign means that if we increase $x$ by $\Delta x$, the function's output will actually *decrease* by $6 \cdot \Delta x$. This will be important when we discuss gradient descent.

Now, what if we have a *multi-variable* function of two (or more) inputs, like this one:
$$ f(x_1, x_2) = 3 x_1^2 + 2 x_2 $$

How do we take the derivative in this case, and what do we even mean by the derivative when there are multiple variables? Here's the trick, and we've actually been doing it already with the sliders: *change one variable at a time, holding the others fixed*. The derivative calculated this way is called a **partial derivative**. Let's compute the partial derivatives of $f$ with respect to $x_1$ and $x_2$:
$$\begin{aligned}
\frac{\partial f}{\partial x_1} &= \frac{\partial}{\partial x_1}3 x_1^2 + \frac{\partial}{\partial x_1}2 x_2 = 6 x_1 + 0 = 6 x_1 && \text{($x_2$ is treated just like a constant)} \\
\frac{\partial f}{\partial x_2} &= \frac{\partial}{\partial x_2}3 x_1^2 + \frac{\partial}{\partial x_2}2 x_2 = 0 + 2 = 2 && \text{($x_1$ is treated just like a constant)}
\end{aligned}$$
The curly $\partial$ (read "partial") is just a reminder that the other variables are being held fixed. When we calculate partial derivatives of a function with respect to all of its variables, and stack them together, we get the **gradient** of the function:
$$ \nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \end{bmatrix} = \begin{bmatrix} 6 x_1 \\ 2 \end{bmatrix} $$

The gradient is sometimes written as $\nabla_{\mathbf{x}} f$ to explicitly indicate that we are taking the partial derivatives with respect to the variables $\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$.

> **Note:** When we stack objects like we did above, the result is called a **vector**. This definition is neither formal nor fully precise, but for now, it is sufficient to think of a vector as simply a column of objects.

Now let's see why this is important. Remember how we defined the mean squared error:
$$ \mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 $$
So the loss depends on the predictions of our model, $\hat{y}_i$. But we calculate $\hat{y}_i$ as:
$$ \hat{y}_i = f(x_i) = w_0 + w_1 x_i, $$
so substituting:
$$ \mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (y_i - (w_0 + w_1 x_i))^2 $$

That reveals a very important fact! It's telling us that the loss is essentially a function of the weights $w_0$ and $w_1$ (as we change $w_0$ and $w_1$, we get different values of the loss).

Now let's summarize what we have so far:
>1. The loss is a function of the weights (such as $w_0$ and $w_1$).
>2. The partial derivative of a function (such as the loss function) with respect to a variable (such as a weight) tells us how fast the function's output changes as we nudge that variable.
>3. The gradient of a function is the vector of all its partial derivatives.

It remains to connect all these points with the gradient descent algorithm.

## Gradient descent: rolling downhill
To keep things simple, let's learn just one weight: we keep the bias $w_0$ fixed at the value we fitted earlier and let only the weight $w_1$ change. Call this model $g$:

$$ g(x) = 65,502 + w_1 x $$

We start it off with a *zero-initialized* weight $w_1$. Below is the same slider example, except this time we cannot adjust the bias term (it's fixed). Slowly move the slider for $w_1$ until reaching the other end point, and watch how it forms a valley on the right.

In [ ]:
w0_fixed = w0
start_w1 = 0

fig2, (axL2, axR2) = plt.subplots(1, 2, figsize=(11, 4.3),
                                  gridspec_kw={"width_ratios": [2, 1]})
plt.close(fig2)

history2 = []
canvas2  = widgets.Image(format="png")

w1_slider2 = widgets.FloatSlider(value=start_w1, min=0, max=300, step=5,
                                 description="weight w1", readout_format=".0f", **slider)

def redraw2(_=None):
    history2.append(mse_at(w1_slider2.value, w0_fixed))
    del history2[:-50]
    redraw_axes(axL2, axR2, w1_slider2.value, w0_fixed, history2)
    canvas2.value = render_png(fig2)

w1_slider2.observe(redraw2, names="value")
redraw2()
display(widgets.VBox([w1_slider2, canvas2]))

We have a beautiful and intuitive picture: the loss is a valley, and training means getting to the bottom. Gradient descent is a simple and automatic way to do it. Our random model sits up on the slope of that valley, not necessarily at the bottom. Gradient descent's job is to walk the weight $w_1$ downhill to where the loss is smallest.

But which way is downhill? Should we increase $w_1$ or decrease it? The answer is hiding in the derivative of the loss curve. Recall from the partial-derivatives section that the derivative is just the *slope*: it measures how the loss changes as we nudge $w_1$. So let's stand at two different weights, one to the **left** of the bottom and one to the **right**, and read off the slope at each.

In [ ]:
def loss(w1):
    return np.mean((price - (w0 + w1 * sqft)) ** 2)

def dloss(w1):
    pred = w0 + w1 * sqft
    return np.mean(2 * (pred - price) * sqft)

print(f"At w1 = 60  (left of the minimum):  derivative = {dloss(60):>14,.0f}   ->  -sign(derivative) = {-np.sign(dloss(60)):+.0f}   (step right)")
print(f"At w1 = 240 (right of the minimum): derivative = {dloss(240):>14,.0f}   ->  -sign(derivative) = {-np.sign(dloss(240)):+.0f}   (step left)")

grid   = np.linspace(0, 300, 300)
curve  = [loss(w) for w in grid]
w1_min = np.sum((price - w0) * sqft) / np.sum(sqft ** 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, w1_here, side in [(axes[0], 60, "LEFT"), (axes[1], 240, "RIGHT")]:
    d         = dloss(w1_here)
    direction = -np.sign(d)

    ax.plot(grid, curve, color="tab:purple", lw=2)
    ax.axvline(w1_min, color="gray", lw=1, ls=":")
    ax.text(w1_min + 4, 0.06 * max(curve), "minimum", color="gray",
            ha="left", va="bottom", fontsize=9)
    ax.scatter([w1_here], [loss(w1_here)], color="black", zorder=6)

    x_tail = w1_here - direction * 35
    x_head = w1_here + direction * 35
    y_tail = loss(w1_here) + d * (x_tail - w1_here)
    y_head = loss(w1_here) + d * (x_head - w1_here)
    ax.annotate("", xy=(x_head, y_head), xytext=(x_tail, y_tail),
                arrowprops=dict(arrowstyle="-|>", color="tab:green", lw=2.5, mutation_scale=22))
    ax.text(x_tail, y_tail, "tangent ", color="tab:green", fontweight="bold",
            fontsize=9, ha=("right" if direction > 0 else "left"), va="bottom")

    sign_word = "negative" if d < 0 else "positive"
    step_dir  = "right" if direction > 0 else "left"
    box = (f"derivative = {d:,.0f}\n"
           f"its sign is {sign_word}\n"
           f"-sign(derivative) = {direction:+.0f}\n"
           f"=> step {step_dir}, toward the bottom")
    ax.text(0.5, 0.96, box, transform=ax.transAxes, ha="center", va="top",
            family="monospace", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="whitesmoke", edgecolor="gray"))

    ax.set_title(f"Standing to the {side} of the minimum   (w1 = {w1_here})", fontsize=11)
    ax.set_xlabel("weight  w1")

axes[0].set_ylabel("loss (MSE)")
plt.tight_layout()
plt.show()

Look at what the derivative tells us in each panel:

- **Standing to the left of the bottom** (left panel): the curve slopes *downhill to the right*, so the derivative is *negative*. Then $-\text{sign}(\text{derivative}) = -(-1) = +1$, which says *increase* $w_1$ — step *right*, toward the bottom.
- **Standing to the right of the bottom** (right panel): the curve slopes *uphill to the right*, so the derivative is *positive*. Then $-\text{sign}(\text{derivative}) = -(+1) = -1$, which says *decrease* $w_1$ — step *left*, again toward the bottom.

Notice how $-\text{sign}(\text{derivative})$ always gives us the right direction to step. That is the main idea behind gradient descent. Wherever you stand, $-\text{sign}(\text{derivative})$ points toward the bottom of the valley. Step that way and the loss will go down.

Now we know the *direction* to step. But how big should the step be? Notice in the plots that the slope is steep when we're high up the wall and gets gentler as we near the bottom. The magnitude of the derivative captures this: it's large far from the minimum and shrinks toward zero as we approach the bottom. So instead of always taking a fixed step in the $-\text{sign}$ direction, gradient descent steps by an amount proportional to the derivative itself: large steps when far away, smaller steps as it settles in (and its sign still points downhill, just as before). Finally, we scale the whole thing by a small positive number $\eta$ (the greek letter "eta"), called the **learning rate**.

**The update rule:** Putting all together, gradient descent simply repeats one step:

$$ w_1^{(t+1)} \;\leftarrow\; w_1^{(t)} \;-\; \eta \, \frac{\partial \mathcal{L}^{(t)}}{\partial w_1} $$

Be aware that superscripts in parentheses (like $w_1^{(t)}$) are not exponents but just notation to keep track of the iteration number $t$ (so $w_1^{(t)}$ means the value of $w_1$ at iteration $t$). Now read it left to right: *new weight = old weight − (learning rate) × (the slope)*. That minus sign is the "go opposite the slope, i.e. downhill" part and it is the $-\text{sign}$ idea from above, now carrying the slope's magnitude as well. The slope $\frac{\partial \mathcal{L}}{\partial w_1}$ is the partial derivative of the loss with respect to the weight (notice that since we have a single weight $w_1$ in this example, we could have written $\frac{d \mathcal{L}}{d w_1}$).

Once we have the update rule for a single weight, we can easily generalize it to multiple weights. If we have weights $w_0, w_1, \dots, w_n$, we just compute the partial derivative of the loss with respect to each weight, create a vector of those derivatives (the gradient), and step downhill in all of them at once. We can write this compactly as:
$$ \mathbf{w}^{(t+1)} \;\leftarrow\; \mathbf{w}^{(t)} \;-\; \eta \, \nabla \mathcal{L}^{(t)} $$
The boldface $\mathbf{w} = \begin{bmatrix} w_0 \\ w_1 \\ \vdots \\ w_n \end{bmatrix}$ is a vector of all weights, and $\nabla \mathcal{L} = \begin{bmatrix} \frac{\partial \mathcal{L}}{\partial w_0} \\ \frac{\partial \mathcal{L}}{\partial w_1} \\ \vdots \\ \frac{\partial \mathcal{L}}{\partial w_n} \end{bmatrix}$ is the gradient of the loss with respect to all those weights. So this update rule can be simply read as:
$$\begin{aligned} 
w_0^{(t+1)} &\leftarrow w_0^{(t)} - \eta \, \frac{\partial \mathcal{L}^{(t)}}{\partial w_0} \\
w_1^{(t+1)} &\leftarrow w_1^{(t)} - \eta \, \frac{\partial \mathcal{L}^{(t)}}{\partial w_1} \\
&\vdots \\
w_n^{(t+1)} &\leftarrow w_n^{(t)} - \eta \, \frac{\partial \mathcal{L}^{(t)}}{\partial w_n}
\end{aligned}$$

Note that it's not a one-time update. Gradient descent is an iterative algorithm, meaning we repeat the above update rule for many times. Repeat the step over and over and $w_1$ slides down to the bottom of the valley.

Let's watch it happen from two angles at once: on the **left**, the weight $w_1$ rolling down the valley; on the **right**, the loss (MSE) dropping step by step as training proceeds.

In [ ]:
def mse_of(w1_val):
    return np.mean((price - (w0 + w1_val * sqft)) ** 2)

def gradient(w1_val):
    pred = w0 + w1_val * sqft
    return np.mean(2 * (pred - price) * sqft)

lr = 1e-7
w1_step = 0.0
history = [w1_step]
for _ in range(8):
    w1_step = w1_step - lr * gradient(w1_step)
    history.append(w1_step)

print("w1 over the steps:", [round(float(h), 1) for h in history])
print("loss (MSE) over the steps:", [round(float(mse_of(h)), 0)/1e11 for h in history])

step_losses = [mse_of(h) for h in history]
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 4))

grid = np.linspace(0, 300, 200)
ax_left.plot(grid, [mse_of(w) for w in grid], color="tab:purple", lw=2, label="loss valley")
ax_left.plot(history, step_losses, "o-", color="tab:red", label="gradient-descent steps")
ax_left.scatter([history[0]], [step_losses[0]], color="black", zorder=6, label="start")
ax_left.set_xlabel("weight  w1"); ax_left.set_ylabel("loss (MSE)")
ax_left.set_title("Rolling down the valley")
ax_left.legend()

ax_right.plot(range(len(step_losses)), step_losses, "o-", color="tab:red")
ax_right.set_xlabel("step (time)"); ax_right.set_ylabel("loss (MSE)")
ax_right.set_title("Loss drops as training proceeds")

plt.tight_layout()
plt.show()

### What happens with two weights?

Until now we kept the bias $w_0$ frozen and let only $w_1$ move, so the loss traced out a curve (the valley you just rolled down). But a real model has many weights and tunes them all together. So let's set both $w_0$ and $w_1$ free.

With two knobs instead of one, the loss is no longer a curve but a surface: every pair $(w_0, w_1)$ gives some loss, and using that loss as the height above the $(w_0, w_1)$ plane carves out a bowl. Gradient descent behaves the same as before, except now it moves in both weights at once, rolling down the bowl to the lowest point.

Watch for the same behavior we saw in the 1D valley: big steps high up where the bowl is steep, shrinking to tiny steps as it settles into the flat bottom. In the plot below, you can move the view with your mouse.

In [ ]:
import plotly.graph_objects as go

z = (sqft - sqft.mean()) / sqft.std()

def loss2(w0_, w1_):
    return np.mean((price - (w0_ + w1_ * z)) ** 2)

def grad2(w0_, w1_):
    err = (w0_ + w1_ * z) - price
    return np.mean(2 * err), np.mean(2 * err * z)

w0_star, w1_star = price.mean(), np.mean(price * z)
R = 260_000
A = np.linspace(w0_star - R, w0_star + R, 80)
B = np.linspace(w1_star - R, w1_star + R, 80)
GA, GB = np.meshgrid(A, B)
SURF = np.array([[loss2(a, b) for a, b in zip(ra, rb)] for ra, rb in zip(GA, GB)])

lr = 0.15
w0c, w1c = w0_star - 200_000, w1_star + 160_000
path0, path1, pathL = [w0c], [w1c], [loss2(w0c, w1c)]
for _ in range(12):
    g0, g1 = grad2(w0c, w1c)
    w0c, w1c = w0c - lr * g0, w1c - lr * g1
    path0.append(w0c); path1.append(w1c); pathL.append(loss2(w0c, w1c))

lift = 0.02 * (SURF.max() - loss2(w0_star, w1_star))
pathLd = [v + lift for v in pathL]

fig = go.Figure()
fig.add_surface(x=A, y=B, z=SURF, colorscale="Viridis", opacity=0.85, showscale=False,
                name="loss surface")
fig.add_scatter3d(x=path0, y=path1, z=pathLd, mode="lines+markers",
                  line=dict(color="red", width=5), marker=dict(size=4, color="red"),
                  name="gradient-descent steps")
fig.add_scatter3d(x=[path0[0]], y=[path1[0]], z=[pathL[0] + lift], mode="markers",
                  marker=dict(size=6, color="black"), name="start")
fig.add_scatter3d(x=[w0_star], y=[w1_star], z=[loss2(w0_star, w1_star) + lift], mode="markers",
                  marker=dict(size=8, color="gold", symbol="diamond",
                              line=dict(color="black", width=1)), name="bottom (lowest loss)")
fig.update_layout(
    title="Two weights: the loss is a bowl",
    height=560, margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(
        xaxis=dict(title="weight w0", showticklabels=False),
        yaxis=dict(title="weight w1", showticklabels=False),
        zaxis=dict(title="loss (MSE)", showticklabels=False),
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
    ),
)
fig.show()

The same idea extends to models with many more weights, although we can only visualize up to three dimensions. In practice, modern models contain millions or even billions of parameters, and their loss landscapes are unimaginably complex. Nevertheless, the same gradient descent algorithm is used to train even the most sophisticated models.

### Why the learning rate matters

The learning rate $\eta$ controls the step size, and getting it wrong hurts. Below we compare three rates on the same valley:

- a **way-too-small** rate takes steps so tiny that after the same number of updates it has barely left the start,
- a **good** rate slides smoothly down to the bottom,
- a **way-too-large** rate overshoots harder every step, so the weight is flung farther and farther out and the loss explodes. This is called **divergence**, and training fails completely.

In [ ]:
def descent_path(learning_rate, steps=8, start=0.0):
    w, path = start, [start]
    for _ in range(steps):
        w = w - learning_rate * gradient(w)
        path.append(w)
    return path

small_path   = descent_path(3e-9)
good_path    = descent_path(1.0e-7)
diverge_path = descent_path(3e-7)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.9))

grid   = np.linspace(-50, 340, 200)
valley = [mse_of(w) for w in grid]
for ax, path, title in [(axes[0], small_path, "way too small (3e-9): barely moves"),
                        (axes[1], good_path,  "good rate (1e-7): finds the bottom")]:
    ax.plot(grid, valley, color="tab:purple", lw=2)
    ax.plot(path, [mse_of(w) for w in path], "o-", color="tab:red")
    ax.scatter([path[0]], [mse_of(path[0])], color="black", zorder=6, label="start")
    ax.scatter([path[-1]], [mse_of(path[-1])], color="tab:green", marker="*", s=220,
               edgecolor="black", zorder=7, label="end")
    ax.set_xlim(-50, 340); ax.set_ylim(0, max(valley) * 1.05)
    ax.set_xlabel("weight  w1"); ax.set_title(title, fontsize=10); ax.legend()
axes[0].set_ylabel("loss (MSE)")

wide = np.linspace(min(diverge_path) - 80, max(diverge_path) + 80, 300)
axes[2].plot(wide, [mse_of(w) for w in wide], color="tab:purple", lw=2)
axes[2].plot(diverge_path, [mse_of(w) for w in diverge_path], "o-", color="tab:red")
axes[2].scatter([diverge_path[0]], [mse_of(diverge_path[0])], color="black", zorder=6, label="start")
axes[2].scatter([diverge_path[-1]], [mse_of(diverge_path[-1])], color="tab:green", marker="*", s=220,
                edgecolor="black", zorder=7, label="end")
axes[2].set_xlabel("weight  w1"); axes[2].set_ylabel("loss (MSE)")
axes[2].set_title("way too large (3e-7): diverges!", fontsize=10); axes[2].legend()

plt.tight_layout()
plt.show()

Parameters that we set manually, such as the learning rate, are called **hyperparameters**. Unlike model weights, they are not optimized automatically by gradient descent and that part is up to us. In real-world training, it's common to deal with dozens of hyperparameters. That said, choosing their values isn't just guesswork. There's a systematic approach called **hyperparameter tuning**, which we'll explore later.

> **Hint:** Go to https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent-exercise and try the interactive learning-rate exercise. Try these learning rates (don't forget to `reset` each time!):
- $\eta_1 = 1.00 \times 10^{-8}$ (the left end of the slider),
- $\eta_2 = 3.00 \times 10^{-2}$ (closer to the right end),
- $\eta_3 = 1.00$ (the right end of the slider).

See how it matches the behavior we just described: $\eta_1$ is way too small and takes forever to get anywhere; $\eta_2$ is just right and slides down smoothly; $\eta_3$ is way too large and diverges.

As a final word: In this notebook, we have dealt with smooth landscapes where the loss is a nice bowl. This is most often not the case. Real-world loss curves are often very bumpy, with many hills and valleys. Thus, we do not have a guarantee that gradient descent will find the **global minimum**. The plot below shows how important initialization is in such cases.

In [ ]:
from matplotlib.lines import Line2D

rng_bumpy = np.random.default_rng(7)

def make_curve(rng):
    K = 3
    A = rng.uniform(0.5, 1.1, K)
    b = rng.uniform(0.7, 1.8, K)
    c = rng.uniform(0, 2 * np.pi, K)
    loss  = lambda w: 0.12 * w**2 + sum(A[k] * np.sin(b[k] * w + c[k]) for k in range(K))
    dloss = lambda w: 0.24 * w    + sum(A[k] * b[k] * np.cos(b[k] * w + c[k]) for k in range(K))
    return loss, dloss

def roll_downhill(dloss, start, lr=0.08, steps=80):
    w = start
    for _ in range(steps):
        w = w - lr * dloss(w)
    return w

for _ in range(3):
    loss, dloss = make_curve(rng_bumpy)

W = np.linspace(-6, 6, 600)
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(W, loss(W), color="tab:purple", lw=2)
w_global = W[np.argmin(loss(W))]
ax.scatter([w_global], [loss(w_global)], marker="*", s=320, color="gold",
           edgecolor="black", zorder=6)
for start in np.linspace(-5, 5, 5):
    end = roll_downhill(dloss, start)
    ax.scatter([start], [loss(start)], color="black", s=22, zorder=5)
    stuck = loss(end) > loss(w_global) + 0.1
    ax.scatter([end], [loss(end)], s=55, zorder=9, edgecolor="black", linewidth=0.5,
               color="tab:red" if stuck else "tab:green")
ax.set_xlabel("weight"); ax.set_ylabel("loss"); ax.set_title("a random bumpy loss", fontsize=10)

handles = [Line2D([0], [0], marker="o", color="black", ls="None", ms=6, label="start"),
           Line2D([0], [0], marker="o", color="tab:red", mec="black", ls="None", ms=8, label="stuck in a local min"),
           Line2D([0], [0], marker="o", color="tab:green", mec="black", ls="None", ms=8, label="reached the global min"),
           Line2D([0], [0], marker="*", color="gold", mec="black", ls="None", ms=14, label="global min")]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=True, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Gradient descent rolls to the nearest valley, so it can get stuck in a local minimum", fontsize=11)
plt.tight_layout(rect=[0, 0.1, 1, 0.93])
plt.show()

## Summary

- A **linear model** has the form $f(x_1,\dots,x_n) = w_0 + w_1 x_1 + \dots + w_n x_n$. The $w_i$ are **weights** (how much each feature counts) and $w_0$ has a special name **bias** (a constant offset that shifts the line).
- To a computer, a "good" model is one with **low loss**. The **squared error** $(y - \hat{y})^2$ scores one prediction; the **Mean Squared Error** $\frac{1}{N}\sum (y_i - \hat{y}_i)^2$ averages it over the whole dataset.
- Plotting loss against a weight reveals a valley. **Gradient descent** reaches the bottom by repeatedly stepping opposite the slope: $w \leftarrow w - \eta\, \frac{\partial \mathcal{L}}{\partial w}$, where the **learning rate** $\eta$ sets the step size.